# TATN 预训练阶段完整复现
**目标**：对 5 个温度（25°C、10°C、0°C、-10°C、-20°C）分别预训练，保存模型和结果，对照论文 Table II / III。

**运行前**：Runtime → Change runtime type → GPU

## Step 1：挂载 Google Drive，设置路径

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ===== 根据你的 Drive 结构修改这两个路径 =====
DRIVE_RAW_DATA = '/content/drive/MyDrive/Research/mining review/TATN/Panasonic 18650PF Data'
DRIVE_CODE_PATH = '/content/drive/MyDrive/Research/mining review/TATN'
WORK_DIR = '/content/TATN'
# =============================================

print('原始数据目录：')
for item in sorted(os.listdir(DRIVE_RAW_DATA)):
    print(' ', item)

## Step 2：拉取代码

In [ ]:
import shutil
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
# 方式A：从你的 fork 拉（推荐）
!git clone https://github.com/zhangcastle/TATN.git /content/TATN
# 方式B：如果代码在 Drive 里，改用下面这行
# shutil.copytree(DRIVE_CODE_PATH, WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
print('工作目录:', os.getcwd())

## Step 3：安装依赖

In [ ]:
!pip install scipy tqdm scikit-learn -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 4：探索原始数据文件结构

In [ ]:
# 运行后确认每个温度都找到了 .mat 文件
temp_folders = {
    '25':  os.path.join(DRIVE_RAW_DATA, 'Panasonic 18650PF Data', '25degC', 'Drive cycles'),
    '10':  os.path.join(DRIVE_RAW_DATA, '10degC', 'Drive cycles'),
    '0':   os.path.join(DRIVE_RAW_DATA, '0degC',  'Drive cycles'),
    'n10': os.path.join(DRIVE_RAW_DATA, '-10degC', 'Drive cycles'),
    'n20': os.path.join(DRIVE_RAW_DATA, '-20degC', 'Drive cycles'),
}

for temp, path in temp_folders.items():
    print(f'\n=== {temp} ({path}) ===')
    if os.path.exists(path):
        for f in sorted(os.listdir(path)):
            if f.endswith('.mat'):
                print(' ', f)
    else:
        print('  [路径不存在，请修改 temp_folders]')

## Step 5：数据处理
处理所有温度的原始数据 → `normalized_data/Pan/{temp}/`

In [ ]:
import scipy.io as sio
import numpy as np

CYCLE_KEYWORDS = {
    'Cycle_1': ['Cycle_1','Cycle1'],
    'Cycle_2': ['Cycle_2','Cycle2'],
    'Cycle_3': ['Cycle_3','Cycle3'],
    'Cycle_4': ['Cycle_4','Cycle4'],
    'NN':      ['_NN_','_NN.'],
    'US06':    ['US06'],
    'HWFET':   ['HWFET'],
    'LA92':    ['LA92'],
    'UDDS':    ['UDDS'],
}

def match_cycle(fname):
    for cycle, kws in CYCLE_KEYWORDS.items():
        for kw in kws:
            if kw in fname:
                return cycle
    return None

def read_mat(fpath):
    data = sio.loadmat(fpath)
    if 'meas' in data:
        m = data['meas']
        return (m['Time'][0][0].flatten(), m['Current'][0][0].flatten(),
                m['Voltage'][0][0].flatten(), m['Battery_Temp_degC'][0][0].flatten(),
                m['Ah'][0][0].flatten())
    return (data['time'].flatten(), data['current'].flatten(),
            data['voltage'].flatten(), data['temp'].flatten(), data['ah'].flatten())

def process_temp(raw_path, temp_label, out_base, interval=10, train_ratio=0.3):
    out_path = os.path.join(out_base, temp_label)
    os.makedirs(out_path, exist_ok=True)
    mat_files = [f for f in os.listdir(raw_path) if f.endswith('.mat')]
    all_data = {}
    for fname in mat_files:
        cycle = match_cycle(fname)
        if cycle is None:
            print(f'  [警告] 无法识别: {fname}')
            continue
        t, cur, vol, tmp, ah = read_mat(os.path.join(raw_path, fname))
        cur, vol, tmp, ah = cur[::interval], vol[::interval], tmp[::interval], ah[::interval]
        # 找连续段起点
        t2 = t[::interval]
        idx = next((i for i in range(len(t2)-1) if t2[i+1]-t2[i] < 2), 0)
        all_data[cycle] = (cur[idx:], vol[idx:], tmp[idx:], ah[idx:])
        print(f'  [{temp_label}] {fname} -> {cycle}  n={len(ah)-idx}')
    if not all_data:
        print(f'[{temp_label}] 无数据，跳过'); return
    # 计算全局 min/max（按温度，论文方法）
    def gminmax(key_idx):
        vals = np.concatenate([v[key_idx] for v in all_data.values()])
        return vals.min(), vals.max()
    ranges = [gminmax(i) for i in range(4)]
    print(f'  ranges: cur={ranges[0]} vol={ranges[1]} tmp={ranges[2]} ah={ranges[3]}')
    norm = lambda x, r: (x - r[0]) / (r[1] - r[0])
    for cycle, (cur, vol, tmp, ah) in all_data.items():
        cn,vn,tn,an = norm(cur,ranges[0]),norm(vol,ranges[1]),norm(tmp,ranges[2]),norm(ah,ranges[3])
        sp = int(len(an) * train_ratio)
        for suf, sl in [('train', slice(None,sp)), ('test', slice(sp,None))]:
            sio.savemat(os.path.join(out_path, f'{temp_label}{cycle}_{suf}.mat'),
                        {'current':cn[sl].reshape(-1,1),'voltage':vn[sl].reshape(-1,1),
                         'temp':tn[sl].reshape(-1,1),'ah':an[sl].reshape(-1,1)})
        print(f'  [{temp_label}] saved {cycle}  train={sp} test={len(an)-sp}')
    print(f'[{temp_label}] done -> {out_path}\n')

OUTPUT_BASE = os.path.join(WORK_DIR, 'normalized_data', 'Pan')
for temp_label, raw_path in temp_folders.items():
    print(f'=== 处理 {temp_label} ===')
    if os.path.exists(raw_path):
        process_temp(raw_path, temp_label, OUTPUT_BASE)
    else:
        print(f'路径不存在，跳过')
print('全部处理完成')

## Step 6：预训练（5 个温度）

In [ ]:
import sys, time
import torch.nn as nn
import torch.optim as optim
sys.path.insert(0, WORK_DIR)
import importlib
import mydata, models
from pretrain import pretrain
importlib.reload(mydata)

mydata.Pan_data_path = OUTPUT_BASE + '/'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH_SIZE, LR, EVAL_INTERVAL = 2000, 64, 0.001, 200
TEMPS = ['25', '10', '0', 'n10', 'n20']
print(f'Device: {DEVICE}  Epochs: {EPOCHS}')

In [ ]:
for temp in TEMPS:
    print(f'\n{"="*55}\n预训练: {temp}°C\n{"="*55}')
    if not os.path.exists(os.path.join(OUTPUT_BASE, temp)):
        print('数据不存在，跳过'); continue
    importlib.reload(models)
    mdls = {
        'conv': models.conv(), 'lstm': models.lstm(),
        'fc': models.fc(), 'regression': models.regression(),
        'conv_s': models.conv(), 'lstm_s': models.lstm(),
        'fc_s': models.fc(), 'regression_s': models.regression(),
        'discriminator': models.Discriminator(),
    }
    opts = {k: optim.Adam(mdls[k].parameters(), lr=LR)
            for k in ['conv','lstm','fc','regression','discriminator']}
    t0 = time.time()
    pretrain(
        rundir=f'pretrain_{temp}',
        source_temp=temp, target_temp=temp,
        source_data_path=OUTPUT_BASE + '/',
        source_train_set=mydata.Pan_train_set,
        source_test_set=mydata.Pan_test_set,
        models=mdls, criterion=nn.MSELoss(reduction='sum'),
        optimizers=opts, batch_size=BATCH_SIZE, epochs=EPOCHS,
        eval_interval=EVAL_INTERVAL, seed=100,
        device_type=DEVICE, ifsave=True, load_model=False,
    )
    print(f'[{temp}°C] 完成，耗时 {(time.time()-t0)/60:.1f} min')

## Step 7：保存模型到 Google Drive

In [ ]:
import glob
DRIVE_RESULTS = '/content/drive/MyDrive/Research/mining review/TATN/results/pretrain'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

for temp in TEMPS:
    candidates = glob.glob(os.path.join(WORK_DIR, f'run/pretrain_{temp}*/saved_model/best.pt'))
    if candidates:
        dest = os.path.join(DRIVE_RESULTS, f'pre-{temp}.pt')
        shutil.copy(candidates[0], dest)
        print(f'保存: pre-{temp}.pt ({os.path.getsize(dest)/1024/1024:.1f} MB)')
    else:
        print(f'[未找到] {temp}°C best.pt')
print('完成')